# 19 — Why are correct same-problem prefixes close? A text-similarity audit

**First hypothesis:** correct attempts at the same problem may have more similar generated prefixes, and that similarity may be associated with closer `pre_last` representations.

Notebook 18 established a contribution of same-problem neighborhoods to the measured ID gap. It did not establish what creates those neighborhoods. This notebook starts with observable text properties, not a claim about a reasoning mechanism.

## Questions fixed before inspecting this audit

1. Within the same problems, are correct–correct prefixes more similar than incorrect–incorrect prefixes in token-bigram overlap?
2. Does any similarity difference persist descriptively after masking numerical values in a separate word/punctuation comparison?
3. Within each problem and class, do more similar pairs tend to have smaller normalized `pre_last` distances?
4. Do last-token identity, short shared suffixes, length differences or earlier occurrences of the selected numeric value provide alternative clues?

Expected directions for the first hypothesis: CC similarity greater than II similarity; CC distance smaller than II distance; negative within-problem/class similarity–distance correlations. All are descriptive predictions. This notebook does not estimate a causal mediation effect or show that textual overlap fully explains the ID findings.

## A different unit of analysis

Use the same filtered source cohort, but restrict the pair comparison to **88 problems with at least two correct AND two incorrect attempts**. Compare all unordered same-class pairs: **1,056 CC pairs and 10,084 II pairs**. Compute means within each problem/class first, then give problems equal weight. Problems with only one correct attempt cannot contribute a CC pair and are reported in the retention table.

These are not the earlier 397-point matched clouds. Pairs reuse attempts, so they are not independent samples. There are no pair-level significance tests or confidence intervals. Missing features retain missing values; finite support is reported rather than replaced with zeros.

## Text boundaries and scope

The notebook verifies the exact stored full-prefix token hashes and uses the same selected-span cut as the extraction. Text similarity excludes the common prompt. Tokens crossing the prompt boundary are omitted from token comparisons and counted. It loads the tokenizer only, not Phi-2.

Token-bigram set Jaccard is the primary text metric. Numerically masked lowercase word/punctuation-bigram Jaccard is secondary, alongside prefix/suffix overlap, last-token identity, numeric-value overlap and length difference. These operational measures do not directly identify shared reasoning steps.

Earlier occurrences of the selected numeric value are retrospective text diagnostics. They may be intermediate values, not earlier final answers. No ground-truth-answer interpretation or prospective prediction claim is made.

## Colab execution

1. Upload this standalone notebook. CPU is sufficient.
2. Leave QUICK=True and RUN_DISTANCES=True. Run numbered cells 1–7 in order.
3. If SOURCE13 is not found automatically, set it to the original FINAL 13 directory or its extracted review bundle. The existing prefix cache and frozen parquet must remain in Drive.
4. Send the QUICK review_bundle.zip. QUICK uses **all text pairs**, but geometric distances only at L0/L24/L31. It never substitutes those layers for L24–L31.
5. After review, set QUICK=False and rerun cells 1–7 for all 32 layers and the fixed L24–L31 summary.

No previous notebook needs rerunning, and no new generation/extraction is required. Standard dependencies are NumPy, pandas, torch, transformers, matplotlib, tqdm and a parquet engine. If missing in your runtime, install them in a separate Colab cell with `%pip install numpy pandas torch transformers matplotlib tqdm pyarrow`, then restart if Colab requests it.

Distance checkpoints resume per layer with a signed configuration. Token and pair audits rerun on restart, but completed geometric layers are reused. Outputs go to a separate `prefix_similarity_phi2_v1` folder. Software/tokenizer/configuration changes produce a separate signature.

In [ ]:
# 1 — Configuration. This notebook computes text features and pair distances, not ID.
import json,hashlib,platform,gc,itertools,zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import transformers
from tqdm.auto import tqdm
from IPython.display import display
try:
    from google.colab import drive
except ImportError:pass
else:drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/msc_thesis')
SOURCE13=None
QUICK=True
RUN_DISTANCES=True
LAYERS=[0,24,31] if QUICK else list(range(32))
EXPECTED_SOURCE_SHA='2bb21797cc39443a066b47a9afab2bece98d99e08879562d05ca0d0c7df08e74'
PREFIX=BASE/'data/representations/phi2/phi2_preanswer_paired_v1.pt'
FROZEN=BASE/'data/frozen/phi2_dataset_frozen_v1.parquet'
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def bool_values(a):
    s=pd.Series(a).astype(str).str.lower()
    if not s.isin(['true','false','0','1']).all():raise ValueError('Invalid Boolean')
    return s.isin(['true','1']).to_numpy()
CONFIG=dict(version='prefix_similarity_v1',quick=QUICK,layers=LAYERS,view='pre_last',
    pairing='all unordered CC and II pairs, same common eligible problems',
    weight='equal problem weights within class; paired class contrast on common finite support',
    minimum_attempts_per_class=2,late_window=list(range(24,32)),
    primary_similarity='token_bigram_jaccard',secondary_similarity='masked_word_bigram_jaccard',
    dtype='float64',normalize=True,implementation_sha256='1ebf7f7763f3773b061d77ef8cdd46a45242c82785e6f5bfb42a78090f866896')
print('QUICK' if QUICK else 'FINAL','| CPU pair distances; no model loading')


In [ ]:
# 2 — Explicit text features and numerical functions.
import re
from decimal import Decimal
import numpy as np

NUM_RE=re.compile(r'-?\d+\.?\d*')

def number_value(s):
    return Decimal(s)

def jaccard(a,b):
    a,b=set(a),set(b)
    return len(a&b)/len(a|b) if a|b else np.nan

def bigrams(x):return list(zip(x[:-1],x[1:]))

def prefix_fraction(a,b):
    if not a or not b:return np.nan
    n=0
    for x,y in zip(a,b):
        if x!=y:break
        n+=1
    return n/min(len(a),len(b))

def pair_features(a,b,ta,tb):
    # All sequences/text below contain generated-prefix content only, not prompt tokens.
    wa=re.findall(r'\w+|[^\w\s]',NUM_RE.sub(' NUM ',ta).lower())
    wb=re.findall(r'\w+|[^\w\s]',NUM_RE.sub(' NUM ',tb).lower())
    na=[number_value(x) for x in NUM_RE.findall(ta)]
    nb=[number_value(x) for x in NUM_RE.findall(tb)]
    return dict(token_bigram_jaccard=jaccard(bigrams(a),bigrams(b)),
        common_prefix_fraction=prefix_fraction(a,b),
        last8_common_suffix_fraction=prefix_fraction(list(reversed(a[-8:])),list(reversed(b[-8:]))),
        last_token_equal=float(a[-1]==b[-1]) if a and b else np.nan,
        masked_word_bigram_jaccard=jaccard(bigrams(wa),bigrams(wb)),
        numeric_value_jaccard=jaccard(na,nb),
        generated_tokens_mean=(len(a)+len(b))/2,
        abs_log_length_difference=abs(np.log1p(len(a))-np.log1p(len(b))))

def extract_prefix(text,prompt,ids,offsets,cut,answer_text):
    if not text.startswith(prompt):raise ValueError('Prompt is not an exact prefix of stored text')
    if not 0<cut<len(ids):raise ValueError('Invalid stored cut')
    spans=list(NUM_RE.finditer(text))
    if not spans:raise ValueError('No selected numeric span')
    span=spans[-1]
    overlap=[j for j,(a,b) in enumerate(offsets) if b>span.start() and a<span.end() and b>a]
    if not overlap or overlap[0]!=cut or span.start()<len(prompt) or span.group()!=str(answer_text):
        raise ValueError('Selected numeric span differs from extraction')
    # Same character cut as notebook 15. A token overlapping prompt boundary is excluded
    # from token-similarity features so no prompt content enters those features.
    end=max(len(prompt),offsets[cut][0]);prefix=text[len(prompt):end]
    generated=[ids[j] for j,(a,b) in enumerate(offsets[:cut]) if a>=len(prompt) and b>a]
    crossing=sum(a<len(prompt)<b for a,b in offsets[:cut])
    mentions=sum(number_value(m.group())==number_value(str(answer_text)) for m in NUM_RE.finditer(prefix))
    return generated,prefix,dict(generated_prefix_tokens=len(generated),prompt_boundary_crossing_tokens=crossing,
        prior_selected_value_mentions=mentions,has_prior_selected_value=int(mentions>0),
        numeric_mentions=len(NUM_RE.findall(prefix)))

def normalize_rows(x):
    x=np.asarray(x,np.float64)
    if not np.isfinite(x).all():raise FloatingPointError('Invalid vectors')
    n=np.linalg.norm(x,axis=1,keepdims=True)
    if (n<=1e-12).any():raise FloatingPointError('Zero vector norm')
    return x/n

def pair_distances(x,a,b,batch=512):
    x=normalize_rows(x);out=np.empty(len(a),np.float64)
    for start in range(0,len(a),batch):
        sl=slice(start,start+batch);out[sl]=np.linalg.norm(x[a[sl]]-x[b[sl]],axis=1)
    if not np.isfinite(out).all() or (out<=1e-12).any():raise FloatingPointError('Invalid/duplicate pair; do not filter automatically')
    return out

def rank_correlation(x,y):
    # Caller supplies pandas Series for tied average ranks; no SciPy dependency.
    valid=x.notna()&y.notna();x=x[valid];y=y[valid]
    if len(x)<5 or x.nunique()<2 or y.nunique()<2:return np.nan
    return float(np.corrcoef(x.rank(method='average'),y.rank(method='average'))[0,1])


In [ ]:
# 3 — Reconstruct the exact extracted prefixes and verify their stored token hashes.
if SOURCE13 is None:
    candidates=list((BASE/'results/aligned_four_views_phi2_v1/final').glob('*/manifest.json'))
    matches=[p.parent for p in candidates if digest(p)==EXPECTED_SOURCE_SHA]
    if len(matches)!=1:raise ValueError('Set SOURCE13 to the original FINAL 13 run or its extracted review bundle')
    SOURCE13=matches[0]
SOURCE13=Path(SOURCE13)
assert digest(SOURCE13/'manifest.json')==EXPECTED_SOURCE_SHA
source=json.loads((SOURCE13/'manifest.json').read_text())
for path in [PREFIX,FROZEN]:
    expected=[v for k,v in source['inputs'].items() if Path(k).name==path.name]
    assert len(expected)==1 and digest(path)==expected[0],f'Input changed: {path}'
meta=pd.read_csv(SOURCE13/'cohort_membership.csv')
assert np.array_equal(meta[['problem_id','attempt_id']].to_numpy(),source['common_keys'])
clean=np.asarray(source['clean_indices'],int)
assert np.array_equal(clean,np.flatnonzero(~bool_values(meta.excluded)))
META=meta.iloc[clean].reset_index(drop=True).copy();META['correct']=bool_values(META.correct)
KEYS=META[['problem_id','attempt_id']].to_numpy(int)
assert len(META)==2541 and META.correct.sum()==433
frozen=pd.read_parquet(FROZEN).set_index(['problem_id','attempt_id']);assert frozen.index.is_unique
audit=pd.read_csv(SOURCE13/'tokenization_audit.csv').set_index(['problem_id','attempt_id']);assert audit.index.is_unique
cache=torch.load(PREFIX,map_location='cpu',weights_only=False) # trusted original extraction cache
lookup={(int(r['problem_id']),int(r['attempt_id'])):r for r in cache};assert len(lookup)==len(cache)
revision=source.get('tokenizer_commit')
tokenizer=AutoTokenizer.from_pretrained('microsoft/phi-2',use_fast=True,**({'revision':revision} if revision else {}))
assert tokenizer.is_fast
TOKENS=[];TEXTS=[];attempt_rows=[];TENSORS=[]
for idx,key in enumerate(tqdm(list(map(tuple,KEYS)),desc='Exact prefix audit')):
    r=lookup[key];row=frozen.loc[key]
    assert bool_values([row.correct])[0]==META.correct.iloc[idx]==bool_values([r['correct']])[0]
    assert tuple(r['pre_last'].shape)==(32,2560)
    text=str(row.generated_text);prompt=str(row.prompt)
    enc=tokenizer(text,add_special_tokens=False,truncation=True,max_length=512,return_offsets_mapping=True)
    ids=enc['input_ids'];cut=int(r['prefix_T'])
    assert len(ids)==int(META.T_full.iloc[idx])==int(r['T_full']) and cut==int(META.prefix_T.iloc[idx])
    h=hashlib.sha256(np.asarray(ids[:cut],dtype=np.int64).tobytes()).hexdigest()
    assert bool_values([audit.loc[key,'ok']])[0] and h==audit.loc[key,'prefix_token_hash'],f'Prefix tokens differ: {key}'
    tokens,prefix,features=extract_prefix(text,prompt,ids,enc['offset_mapping'],cut,r['answer_text'])
    TOKENS.append(tokens);TEXTS.append(prefix);TENSORS.append(r['pre_last'])
    attempt_rows.append(dict(row_index=idx,problem_id=int(key[0]),attempt_id=int(key[1]),correct=bool(META.correct.iloc[idx]),
        prefix_token_hash=h,**features))
ATTEMPTS=pd.DataFrame(attempt_rows)
del lookup,cache,frozen;gc.collect()
print('Verified exact prefix hashes:',len(ATTEMPTS))
print('Prompt-boundary crossing tokens excluded from token features:',ATTEMPTS.prompt_boundary_crossing_tokens.sum())


In [ ]:
# 4 — Common pair cohort; all unordered within-class pairs, then equal problem weighting.
counts=META.groupby(['problem_id','correct']).size().unstack(fill_value=0)
eligible=counts.index[(counts[True]>=2)&(counts[False]>=2)].to_numpy(int)
assert len(eligible)==88, 'Cohort changed: review before proceeding'
retention=counts.rename(columns={False:'incorrect_attempts',True:'correct_attempts'}).reset_index()
retention['eligible']=retention.problem_id.isin(eligible)
pair_rows=[]
for pid in tqdm(eligible,desc='Text similarity for within-problem pairs'):
    for label,kind in [(True,'CC'),(False,'II')]:
        indices=META.index[(META.problem_id==pid)&(META.correct==label)].to_numpy()
        for a,b in itertools.combinations(indices,2):
            fa=ATTEMPTS.iloc[a];fb=ATTEMPTS.iloc[b]
            pair_rows.append(dict(problem_id=int(pid),pair_type=kind,a=int(a),b=int(b),
                attempt_a=int(KEYS[a,1]),attempt_b=int(KEYS[b,1]),
                both_prior_selected_value=int(fa.has_prior_selected_value and fb.has_prior_selected_value),
                either_prior_selected_value=int(fa.has_prior_selected_value or fb.has_prior_selected_value),
                **pair_features(TOKENS[a],TOKENS[b],TEXTS[a],TEXTS[b])))
PAIRS=pd.DataFrame(pair_rows)
assert not PAIRS.duplicated(['a','b']).any()
assert PAIRS.pair_type.value_counts().to_dict()=={'II':10084,'CC':1056}
FEATURES=['token_bigram_jaccard','masked_word_bigram_jaccard','common_prefix_fraction',
    'last8_common_suffix_fraction','last_token_equal','numeric_value_jaccard',
    'generated_tokens_mean','abs_log_length_difference','both_prior_selected_value','either_prior_selected_value']
# Missing similarity remains missing (e.g. neither prefix contains any token bigrams).
problem_features=PAIRS.groupby(['problem_id','pair_type'])[FEATURES].mean()
feature_rows=[]
for feature in FEATURES:
    wide=problem_features[feature].unstack('pair_type').dropna(subset=['CC','II'])
    feature_rows.append(dict(feature=feature,n_common_problems=len(wide),
        CC_mean=wide.CC.mean(),II_mean=wide.II.mean(),CC_minus_II=(wide.CC-wide.II).mean()))
FEATURE_SUMMARY=pd.DataFrame(feature_rows)
# Earlier occurrence of the SELECTED value is a retrospective text diagnostic, not a probe input
# or proof of answer leakage: a matching value can also be an intermediate calculation.
a=ATTEMPTS[ATTEMPTS.problem_id.isin(eligible)].groupby(['problem_id','correct']).has_prior_selected_value.mean().unstack()
MENTIONS=pd.DataFrame([dict(n_problems=len(a),correct_mean=a[True].mean(),incorrect_mean=a[False].mean(),
    correct_minus_incorrect=(a[True]-a[False]).mean())])
MANIFEST=dict(config=CONFIG,source_manifest_sha256=EXPECTED_SOURCE_SHA,
    source_files={str(p):digest(p) for p in [SOURCE13/'cohort_membership.csv',SOURCE13/'tokenization_audit.csv']},
    inputs={str(p):next(v for k,v in source['inputs'].items() if Path(k).name==p.name) for p in [PREFIX,FROZEN]},
    tokenizer_commit=tokenizer.init_kwargs.get('_commit_hash'),
    tokenizer_backend_sha256=hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode()).hexdigest(),
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,torch=torch.__version__,transformers=transformers.__version__),
    n_source_problems=129,n_pair_problems=len(eligible),n_CC_pairs=1056,n_II_pairs=10084,
    pair_features_sha256=hashlib.sha256(PAIRS.to_csv(index=False).encode()).hexdigest())
SIGNATURE=hashlib.sha256(json.dumps(MANIFEST,sort_keys=True).encode()).hexdigest()
OUT=BASE/'results/prefix_similarity_phi2_v1'/('quick' if QUICK else 'final')/SIGNATURE[:12]
OUT.mkdir(parents=True,exist_ok=True);(OUT/'checkpoints').mkdir(exist_ok=True)
(OUT/'manifest.json').write_text(json.dumps(MANIFEST,indent=2))
for name,df in [('retention',retention),('attempt_features',ATTEMPTS),('pair_features',PAIRS),('text_similarity_summary',FEATURE_SUMMARY),('prior_value_mentions_summary',MENTIONS)]:df.to_csv(OUT/(name+'.csv'),index=False)
problem_features.to_csv(OUT/'text_similarity_by_problem.csv')
print('Results:',OUT);print('Common pair cohort:',len(eligible),'problems; all text metrics exclude the prompt')
display(FEATURE_SUMMARY.round(4));display(MENTIONS.round(4))


In [ ]:
# 5 — Distances for fixed pairs. No resampling, no ID estimation, no LLM loading.
if not RUN_DISTANCES:raise RuntimeError('Text audit complete. Set RUN_DISTANCES=True and run cells 5–7.')
A=PAIRS.a.to_numpy(int);B=PAIRS.b.to_numpy(int)
for layer in tqdm(LAYERS,desc='Pair geometry'):
    path=OUT/'checkpoints'/f'L{layer:02d}.npz'
    if path.exists():
        with np.load(path,allow_pickle=False) as z:
            assert str(z['signature'].item())==SIGNATURE
            assert np.array_equal(z['a'],A) and np.array_equal(z['b'],B)
            assert z['distance'].shape==(len(PAIRS),) and np.isfinite(z['distance']).all() and (z['distance']>1e-12).all()
        continue
    raw=np.stack([t[layer].double().numpy() for t in TENSORS])
    distance=pair_distances(raw,A,B)
    temp=path.with_suffix('.tmp')
    with temp.open('wb') as f:np.savez_compressed(f,signature=np.asarray(SIGNATURE),a=A,b=B,distance=distance)
    temp.replace(path);del raw;gc.collect()
print('All requested pair-distance checkpoints complete.')


In [ ]:
# 6 — Equal-problem contrasts and within-problem/class rank correlations.
GEOMETRY=[];PROBLEM_ROWS=[];CORRELATIONS=[];DISTANCES={}
SIMILARITIES=['token_bigram_jaccard','masked_word_bigram_jaccard','common_prefix_fraction','last8_common_suffix_fraction']
def summarize_distances(distance,window):
    f=PAIRS.copy();f['distance']=distance
    means=f.groupby(['problem_id','pair_type']).distance.mean().unstack('pair_type')
    assert len(means)==len(eligible) and means.notna().all().all()
    row=dict(window=window,n_problems=len(means),CC_mean=means.CC.mean(),II_mean=means.II.mean(),
        II_minus_CC=(means.II-means.CC).mean(),fraction_problems_CC_closer=(means.CC<means.II).mean())
    per=means.reset_index();per['window']=window;per['II_minus_CC']=per.II-per.CC
    cor=[]
    for (pid,kind),g in f.groupby(['problem_id','pair_type']):
        for feature in SIMILARITIES:
            value=rank_correlation(g[feature],g.distance)
            cor.append(dict(window=window,problem_id=pid,pair_type=kind,feature=feature,
                n_pairs=len(g),n_finite_pairs=int((g[feature].notna()&g.distance.notna()).sum()),spearman=value))
    return row,per,pd.DataFrame(cor)
for layer in LAYERS:
    path=OUT/'checkpoints'/f'L{layer:02d}.npz'
    if not path.exists():raise RuntimeError(f'Missing checkpoint {path}; resume cell 5')
    with np.load(path,allow_pickle=False) as z:
        assert str(z['signature'].item())==SIGNATURE;DISTANCES[layer]=z['distance'].copy()
    row,per,cor=summarize_distances(DISTANCES[layer],f'L{layer:02d}')
    GEOMETRY.append(row);PROBLEM_ROWS.append(per);CORRELATIONS.append(cor)
HAVE_LATE=set(range(24,32)).issubset(LAYERS)
if HAVE_LATE:
    late_distance=np.mean([DISTANCES[l] for l in range(24,32)],axis=0)
    row,per,cor=summarize_distances(late_distance,'L24-L31')
    GEOMETRY.append(row);PROBLEM_ROWS.append(per);CORRELATIONS.append(cor)
else:print('QUICK: no L24–L31 summary. Only L0, L24 and L31 were computed.')
GEOMETRY=pd.DataFrame(GEOMETRY);PROBLEM=pd.concat(PROBLEM_ROWS);CORR=pd.concat(CORRELATIONS)
# Correlations need >=5 pairs and nonconstant variables. Never replace unavailable correlations by zero.
# Use COMMON finite problem support for CC/II correlation comparisons, reported explicitly.
cor_rows=[]
for (window,feature),g in CORR.groupby(['window','feature']):
    wide=g.pivot(index='problem_id',columns='pair_type',values='spearman').dropna(subset=['CC','II'])
    cor_rows.append(dict(window=window,feature=feature,n_common_problems=len(wide),
        CC_mean_rho=wide.CC.mean(),II_mean_rho=wide.II.mean()))
CORR_SUMMARY=pd.DataFrame(cor_rows)
for name,df in [('geometry_summary',GEOMETRY),('geometry_by_problem',PROBLEM),('correlations_by_problem',CORR),('correlation_summary',CORR_SUMMARY)]:df.to_csv(OUT/(name+'.csv'),index=False)
display(GEOMETRY.round(4));display(CORR_SUMMARY[CORR_SUMMARY.window==('L24-L31' if HAVE_LATE else 'L31')].round(4))
print('Descriptive analysis only: correlated pairs are not independent samples; no pair-level p-values or CIs.')


In [ ]:
# 7 — Figures and review bundle. No raw response text is needed in the bundle.
FIG=OUT/'figures';FIG.mkdir(exist_ok=True)
g=GEOMETRY[GEOMETRY.window!='L24-L31'].copy();g['layer']=g.window.str[1:].astype(int);g=g.sort_values('layer')
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(g.layer,g.CC_mean,label='Correct–correct');axes[0].plot(g.layer,g.II_mean,label='Incorrect–incorrect')
axes[0].set(xlabel='Layer',ylabel='Mean L2-normalized pair distance');axes[0].legend()
axes[1].plot(g.layer,g.II_minus_CC);axes[1].axhline(0,color='gray',lw=.8)
axes[1].set(xlabel='Layer',ylabel='II minus CC mean distance')
fig.suptitle('Equal weighting of 88 common problems | '+('QUICK: selected layers' if QUICK else 'All layers'))
fig.tight_layout();fig.savefig(FIG/'pair_distances.png',dpi=150);plt.show();plt.close(fig)
features=FEATURE_SUMMARY[FEATURE_SUMMARY.feature.isin(SIMILARITIES)]
fig,ax=plt.subplots(figsize=(10,4));x=np.arange(len(features))
ax.bar(x-.18,features.CC_mean,width=.36,label='CC');ax.bar(x+.18,features.II_mean,width=.36,label='II')
ax.set_xticks(x,features.feature,rotation=20,ha='right');ax.set_ylabel('Equal-problem mean similarity');ax.legend()
fig.tight_layout();fig.savefig(FIG/'text_similarity.png',dpi=150);plt.show();plt.close(fig)
# Each scatter point is one PROBLEM mean, not an independent token-pair observation.
window='L24-L31' if HAVE_LATE else 'L31'
fig,ax=plt.subplots(figsize=(7,5))
for kind in ['CC','II']:
    feature=problem_features['token_bigram_jaccard'].unstack('pair_type')[kind]
    geom=PROBLEM[PROBLEM.window==window].set_index('problem_id')[kind]
    joined=pd.concat([feature.rename('similarity'),geom.rename('distance')],axis=1).dropna()
    ax.scatter(joined.similarity,joined.distance,label=kind,alpha=.6)
ax.set(xlabel='Mean generated-prefix token bigram Jaccard',ylabel='Mean pair distance',title=window+' | one point per problem/class');ax.legend()
fig.tight_layout();fig.savefig(FIG/'problem_means_similarity_distance.png',dpi=150);plt.show();plt.close(fig)
notes='''# Prefix text similarity and pair-distance audit

See manifest for QUICK/FINAL. Source cohort: 2541 attempts/129 problems. Common pair cohort: 88 problems with >=2 attempts in EACH class. All 1056 CC and 10084 II unordered within-problem pairs are included. Pair means are computed within each problem/class, then problems receive equal weight. These are not the original 397-point matched clouds and no ID is estimated here.

All text comparisons exclude the shared prompt. Token hashes of full extracted prefixes are verified against the source audit before prompt tokens are removed for text features. Rare tokens crossing the prompt boundary are excluded from token features and counted. Character-based masked-text features use the generated character prefix. Prefix cut and selected numeric span must match the extraction.

Primary similarity: token-bigram set Jaccard. Secondary: lowercase word/punctuation-bigram set Jaccard after replacing numeric spans with NUM. Common-prefix and last-eight-token common-suffix fractions divide by the shorter compared sequence. An empty union/sequence produces missing similarity, not perfect similarity. Support counts are explicit, including common finite problem support in CC/II comparisons.

Earlier selected-value occurrences are retrospective diagnostics matching the numerical value of the selected answer span, excluding the prompt and selected span. They are NOT ground-truth verification, proof of answer leakage or features for a prospective correctness predictor. The inherited numeric parser has limitations. Masking numbers does not establish equivalent mathematical reasoning.

Distances use saved pre_last vectors, normalized in float64. All layers are retained in FINAL and the fixed L24–L31 window averages distances within each pair first. QUICK uses three layers and does not produce a late-window aggregate. No language model or new extraction is used.

Within-problem/class Spearman correlations require >=5 pairs and nonconstant features/distances. Common finite problem support is used when comparing CC and II correlation averages. Pairs share attempts and are not independent observations. No inferential p-values or confidence intervals are provided.

This is an exploratory first audit: no adjustment or causal mediation estimate. Even stronger text similarity and negative similarity-distance correlations would not prove text overlap explains the earlier ID effect. Length, last-token identity, answer mentions and correlated features remain possible explanations. Raw response texts are not included; attempt IDs permit inspection in the frozen data. Missing values are intentional where a statistic is undefined.
'''
(OUT/'REVIEW_NOTES.md').write_text(notes)
with zipfile.ZipFile(OUT/'review_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.glob('*.csv'))+[OUT/'manifest.json',OUT/'REVIEW_NOTES.md']+sorted(FIG.glob('*.png')):
        z.write(path,path.relative_to(OUT))
print('Review bundle:',OUT/'review_bundle.zip')


## Interpretation before designing the next control

- Higher CC text similarity plus smaller CC distances is consistent with a lexical explanation, but does not isolate it.
- Negative correlations within problems/classes show association; correlated length, numbers and token-position effects remain possible.
- Numerically masked overlap remains a surface-form metric, not a test of mathematical equivalence.
- A weak or absent lexical association would motivate other explanations; do not automatically select a more favorable similarity measure.
- The pair cohort excludes singleton-correct problems. Report that limitation when connecting this audit to earlier full-cohort findings.

After this audit, use observed feature support to design an explicit similarity-matched comparison. That follow-up is separate from this notebook; do not interpret the present descriptive results as having already controlled for text similarity.